# SE-801 Lecture 10 — Convolutional Neural-Network Fundamentals

This notebook builds convolution from tensor operations, checks it against
PyTorch, and compares an MLP with a compact CNN on a controlled synthetic
image task. Every run is local, deterministic, and independent of downloads.

In [1]:
from __future__ import annotations

import json
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 20261019
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

print("PyTorch:", torch.__version__)
print("Seed:", SEED)

PyTorch: 2.14.0+cu130
Seed: 20261019


## 1. Cross-correlation from tensor operations

Deep-learning libraries usually implement cross-correlation while retaining
the conventional layer name `Conv2d`. The kernel is not flipped.

In [2]:
def corr2d(x: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    """Valid 2D cross-correlation for one input and one kernel."""
    kh, kw = kernel.shape
    out_h = x.shape[0] - kh + 1
    out_w = x.shape[1] - kw + 1
    y = torch.empty((out_h, out_w), dtype=x.dtype)
    for i in range(out_h):
        for j in range(out_w):
            y[i, j] = (x[i : i + kh, j : j + kw] * kernel).sum()
    return y


image = torch.tensor(
    [
        [0.0, 0.0, 1.0, 1.0, 1.0],
        [0.0, 0.0, 1.0, 1.0, 1.0],
        [0.0, 0.0, 1.0, 1.0, 1.0],
        [0.0, 0.0, 1.0, 1.0, 1.0],
        [0.0, 0.0, 1.0, 1.0, 1.0],
    ]
)
vertical_edge_kernel = torch.tensor([[1.0, -1.0], [1.0, -1.0]])

manual_output = corr2d(image, vertical_edge_kernel)
framework_output = F.conv2d(
    image[None, None], vertical_edge_kernel[None, None]
).squeeze(0).squeeze(0)

correlation_max_error = float((manual_output - framework_output).abs().max())
print("Input image:\n", image)
print("Kernel:\n", vertical_edge_kernel)
print("Cross-correlation output:\n", manual_output)
print("Maximum manual/framework error:", correlation_max_error)

Input image:
 tensor([[0., 0., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 1., 1., 1.]])
Kernel:
 tensor([[ 1., -1.],
        [ 1., -1.]])
Cross-correlation output:
 tensor([[ 0., -2.,  0.,  0.],
        [ 0., -2.,  0.,  0.],
        [ 0., -2.,  0.,  0.],
        [ 0., -2.,  0.,  0.]])
Maximum manual/framework error: 0.0


## 2. Output-shape arithmetic

For one spatial dimension with input size $n$, kernel size $k$, padding $p$,
dilation $d$, and stride $s$:

$$
n_{out}=\left\lfloor\frac{n+2p-d(k-1)-1}{s}\right\rfloor+1.
$$

In [3]:
def conv_out_size(n: int, kernel: int, padding: int = 0, stride: int = 1, dilation: int = 1) -> int:
    return math.floor((n + 2 * padding - dilation * (kernel - 1) - 1) / stride) + 1


shape_cases = [
    {"input": 28, "kernel": 3, "padding": 0, "stride": 1},
    {"input": 28, "kernel": 3, "padding": 1, "stride": 1},
    {"input": 28, "kernel": 5, "padding": 2, "stride": 2},
    {"input": 16, "kernel": 3, "padding": 1, "stride": 2},
]
for case in shape_cases:
    case["output"] = conv_out_size(
        case["input"], case["kernel"], case["padding"], case["stride"]
    )
print("Output-shape checks:")
for case in shape_cases:
    print(case)

Output-shape checks:
{'input': 28, 'kernel': 3, 'padding': 0, 'stride': 1, 'output': 26}
{'input': 28, 'kernel': 3, 'padding': 1, 'stride': 1, 'output': 28}
{'input': 28, 'kernel': 5, 'padding': 2, 'stride': 2, 'output': 14}
{'input': 16, 'kernel': 3, 'padding': 1, 'stride': 2, 'output': 8}


## 3. Controlled image data

The label is orientation: vertical or horizontal. Training and validation bars
appear near the image centre. The test set places bars close to the image
boundary, creating a controlled translation shift. The test set remains frozen
until both models and all hyperparameters are fixed.

In [4]:
def make_bar_images(
    n: int,
    positions: list[int],
    *,
    seed: int,
    image_size: int = 16,
    noise_std: float = 0.22,
) -> tuple[torch.Tensor, torch.Tensor]:
    generator = torch.Generator().manual_seed(seed)
    labels = torch.arange(n) % 2
    permutation = torch.randperm(n, generator=generator)
    labels = labels[permutation]
    images = noise_std * torch.randn(n, 1, image_size, image_size, generator=generator)
    position_ids = torch.randint(0, len(positions), (n,), generator=generator)
    intensities = 0.9 + 0.25 * torch.rand(n, generator=generator)
    for idx, label in enumerate(labels.tolist()):
        pos = positions[int(position_ids[idx])]
        if label == 0:
            images[idx, 0, :, pos : pos + 2] += intensities[idx]
        else:
            images[idx, 0, pos : pos + 2, :] += intensities[idx]
    return images.clamp(-1.0, 1.5), labels.long()


train_x, train_y = make_bar_images(800, list(range(4, 11)), seed=SEED + 1)
val_x, val_y = make_bar_images(200, list(range(4, 11)), seed=SEED + 2)
test_x, test_y = make_bar_images(200, [1, 2, 13, 14], seed=SEED + 3)

print("Train:", tuple(train_x.shape), tuple(train_y.shape))
print("Validation:", tuple(val_x.shape), tuple(val_y.shape))
print("Frozen shifted test:", tuple(test_x.shape), tuple(test_y.shape))

fig, axes = plt.subplots(2, 4, figsize=(9, 4.5), constrained_layout=True)
for row, label in enumerate([0, 1]):
    indices = (train_y == label).nonzero(as_tuple=True)[0][:4]
    for col, idx in enumerate(indices.tolist()):
        axes[row, col].imshow(train_x[idx, 0], cmap="gray", vmin=-1, vmax=1.5)
        axes[row, col].set_title("vertical" if label == 0 else "horizontal")
        axes[row, col].axis("off")
fig.suptitle("Synthetic orientation task")
fig.savefig(OUTPUT_DIR / "synthetic_bar_examples.png", dpi=170)
plt.close(fig)

Train: (800, 1, 16, 16) (800,)
Validation: (200, 1, 16, 16) (200,)
Frozen shifted test: (200, 1, 16, 16) (200,)


## 4. Similar parameter budgets, different assumptions

The MLP assigns independent weights to locations. The CNN reuses each kernel
over the image and pools spatial evidence before classification.

In [5]:
class SmallMLP(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 16, 4),
            nn.ReLU(),
            nn.Linear(4, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SmallCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(8, 12, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(12, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.features(x)
        return self.classifier(z.flatten(1))


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


mlp = SmallMLP()
cnn = SmallCNN()
parameter_counts = {"MLP": count_parameters(mlp), "CNN": count_parameters(cnn)}
print("Trainable parameters:", parameter_counts)

with torch.no_grad():
    sample = train_x[:2]
    shape_trace = [{"stage": "input", "shape": list(sample.shape)}]
    z = sample
    for name, layer in cnn.features.named_children():
        z = layer(z)
        shape_trace.append({"stage": f"features.{name} {layer.__class__.__name__}", "shape": list(z.shape)})
    z = z.flatten(1)
    shape_trace.append({"stage": "flatten", "shape": list(z.shape)})
    z = cnn.classifier(z)
    shape_trace.append({"stage": "classifier", "shape": list(z.shape)})
print("CNN shape trace:")
for row in shape_trace:
    print(row["stage"], row["shape"])

Trainable parameters: {'MLP': 1038, 'CNN': 982}
CNN shape trace:
input [2, 1, 16, 16]
features.0 Conv2d [2, 8, 16, 16]
features.1 ReLU [2, 8, 16, 16]
features.2 MaxPool2d [2, 8, 8, 8]
features.3 Conv2d [2, 12, 8, 8]
features.4 ReLU [2, 12, 8, 8]
features.5 AdaptiveAvgPool2d [2, 12, 1, 1]
flatten [2, 12]
classifier [2, 2]


## 5. Controlled training and validation selection

In [6]:
def accuracy(logits: torch.Tensor, targets: torch.Tensor) -> float:
    return float((logits.argmax(dim=1) == targets).float().mean())


def reset_module(module: nn.Module) -> None:
    if hasattr(module, "reset_parameters"):
        module.reset_parameters()


def train_model(model: nn.Module, *, seed: int, epochs: int = 60) -> tuple[nn.Module, list[dict]]:
    torch.manual_seed(seed)
    model.apply(reset_module)
    loader_generator = torch.Generator().manual_seed(seed + 100)
    train_loader = DataLoader(
        TensorDataset(train_x, train_y),
        batch_size=64,
        shuffle=True,
        generator=loader_generator,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    best_state = None
    best_val_loss = float("inf")
    history: list[dict] = []
    for epoch in range(1, epochs + 1):
        model.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            loss = F.cross_entropy(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_logits = model(train_x)
            val_logits = model(val_x)
            train_loss = float(F.cross_entropy(train_logits, train_y))
            val_loss = float(F.cross_entropy(val_logits, val_y))
            row = {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_accuracy": accuracy(train_logits, train_y),
                "val_accuracy": accuracy(val_logits, val_y),
            }
            history.append(row)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
    assert best_state is not None
    model.load_state_dict(best_state)
    return model, history


mlp, mlp_history = train_model(mlp, seed=SEED + 10)
cnn, cnn_history = train_model(cnn, seed=SEED + 20)

## 6. Frozen shifted test and feature maps

In [7]:
def evaluate(model: nn.Module) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        val_logits = model(val_x)
        test_logits = model(test_x)
    return {
        "validation_loss": float(F.cross_entropy(val_logits, val_y)),
        "validation_accuracy": accuracy(val_logits, val_y),
        "shifted_test_loss": float(F.cross_entropy(test_logits, test_y)),
        "shifted_test_accuracy": accuracy(test_logits, test_y),
    }


metrics = {"MLP": evaluate(mlp), "CNN": evaluate(cnn)}
print(json.dumps(metrics, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for name, history, color in [
    ("MLP", mlp_history, "#B87521"),
    ("CNN", cnn_history, "#246B8E"),
]:
    epochs = [row["epoch"] for row in history]
    axes[0].plot(epochs, [row["val_loss"] for row in history], label=name, color=color)
    axes[1].plot(epochs, [row["val_accuracy"] for row in history], label=name, color=color)
axes[0].set(xlabel="epoch", ylabel="validation cross-entropy", title="Validation loss")
axes[1].set(xlabel="epoch", ylabel="validation accuracy", title="Validation accuracy", ylim=(0.45, 1.02))
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=170)
plt.close(fig)

cnn.eval()
example_index = int((test_y == 0).nonzero(as_tuple=True)[0][0])
example = test_x[example_index : example_index + 1]
with torch.no_grad():
    first_maps = cnn.features[1](cnn.features[0](example))[0]

fig, axes = plt.subplots(2, 4, figsize=(8, 4), constrained_layout=True)
for index, axis in enumerate(axes.flat):
    axis.imshow(first_maps[index], cmap="viridis")
    axis.set_title(f"channel {index}")
    axis.axis("off")
fig.suptitle("First-layer feature maps for a shifted vertical bar")
fig.savefig(OUTPUT_DIR / "first_layer_feature_maps.png", dpi=170)
plt.close(fig)

{
  "MLP": {
    "validation_loss": 3.3110027288785204e-05,
    "validation_accuracy": 1.0,
    "shifted_test_loss": 7.880921840667725,
    "shifted_test_accuracy": 0.0
  },
  "CNN": {
    "validation_loss": 4.2558254790492356e-05,
    "validation_accuracy": 1.0,
    "shifted_test_loss": 0.00030332099413499236,
    "shifted_test_accuracy": 1.0
  }
}


## 7. Reproducibility record

In [8]:
results = {
    "seed": SEED,
    "task": {
        "image_shape": [1, 16, 16],
        "classes": ["vertical", "horizontal"],
        "training_positions": list(range(4, 11)),
        "shifted_test_positions": [1, 2, 13, 14],
        "train_examples": len(train_y),
        "validation_examples": len(val_y),
        "test_examples": len(test_y),
    },
    "cross_correlation_max_error": correlation_max_error,
    "shape_cases": shape_cases,
    "parameter_counts": parameter_counts,
    "shape_trace": shape_trace,
    "metrics": metrics,
    "histories": {"MLP": mlp_history, "CNN": cnn_history},
}

(OUTPUT_DIR / "experiment_results.json").write_text(json.dumps(results, indent=2))
torch.save(cnn.state_dict(), OUTPUT_DIR / "small_cnn_state_dict.pt")
print("Saved results to", OUTPUT_DIR)

Saved results to outputs
